# Diffusion Tensor Imaging Measurements for Computer-Aided Diagnosis of Autism - Derived Data from ABIDE II Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.dbrh-5zc8/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Extract metadata as JSON
metadata = dataset.metadata.to_json()

print(f"Dataset Name: {metadata['name']}")
print(f"Description: {metadata['description']}")
print(f"Published: {metadata['datePublished']}")
print("License:", metadata['license'])
print("Citation:", metadata['citeAs'])
print("Keywords:", metadata['keywords'])

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List available RecordSets and their @id
record_sets = dataset.record_sets
print("Available RecordSets (@id):")
record_set_ids = []
for rs in record_sets:
    print(f"- {rs['@id']}: {rs.get('name', 'No Name')}")
    record_set_ids.append(rs['@id'])

# Show fields and columns for each RecordSet
for rs in record_sets:
    print(f"\nRecordSet: {rs['@id']} ({rs.get('name', 'No Name')})")
    print("Fields:")
    for field in rs.get('field', []):
        print(f"  - {field['@id']} ({field.get('name', 'No Name')})")
        if 'column' in field:
            print("    Columns:")
            for column in field['column']:
                print(f"      * {column['@id']} ({column.get('name', 'No Name')})")

## 3. Data Extraction
Load data from specific record sets into DataFrames for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each RecordSet using @id
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df
    print(f"Loaded DataFrame for RecordSet: {rs_id} - shape: {df.shape}")

# Example: Print column names for one RecordSet
if record_set_ids:
    rs_id_example = record_set_ids[0]
    print(f"\nColumns in {rs_id_example}:")
    print(dataframes[rs_id_example].columns.tolist())
    display(dataframes[rs_id_example].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. Operations may include removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set for EDA (example: first RecordSet)
record_set_id = rs_id_example
df = dataframes[record_set_id]

# Refer to numeric fields by @id - Example: pick FA as field
numeric_candidates = [col for col in df.columns if 'FA' in col or 'fractional' in col or df[col].dtype in ['float64', 'int64']]
if numeric_candidates:
    numeric_field_id = numeric_candidates[0]
else:
    numeric_field_id = df.select_dtypes(include=['float64', 'int64']).columns[0]

threshold = df[numeric_field_id].mean() # Use mean as threshold demonstration
filtered_df = df[df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
print(f"Normalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Try to group by a demographic or region field (example: Age or Gender if available)
group_candidates = [col for col in df.columns if 'age' in col.lower() or 'gender' in col.lower() or 'region' in col.lower()]
if group_candidates:
    group_field_id = group_candidates[0]
    grouped_df = filtered_df.groupby(group_field_id).mean(numeric_only=True)
    print(f"Grouped data by {group_field_id}:")
    display(grouped_df.head())
else:
    print("No suitable grouping field found; skipping group analysis.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Example distribution plot for the normalized numeric field
if numeric_field_id:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"], bins=20, kde=True)
    plt.title(f"Distribution of Normalized {numeric_field_id}")
    plt.xlabel(f"{numeric_field_id}_normalized")
    plt.ylabel("Frequency")
    plt.show()

# If grouped_df exists, visualize group means
if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10,4))
    grouped_df[numeric_field_id].plot(kind='bar')
    plt.title(f"Average {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- Successfully loaded and explored Diffusion Tensor Imaging measurements dataset using the `mlcroissant` library.
- Examined available record sets and fields with unique `@id` references, enabling precise access to data.
- Performed filtering and normalization on relevant numeric fields, and attempted demographic grouping.
- Visualized the distribution and group means for structural brain measurement data.
- Dataset can be used for further machine learning, research, or analytic purposes regarding ASD and DTI metrics.
